Step 1: Import Libraries + Download Dataset + Create Vocabulary

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
import random

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Download Tiny Shakespeare Dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "shakespeare.txt")

print("Dataset Downloaded Successfully!")

Using: cuda
Dataset Downloaded Successfully!


Step 1.2 Read the Dataset

In [ ]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total Characters :", len(text))
print(text[:500])

Total Characters : 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


Step 1.3 Build Character Vocabulary

In [ ]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocabulary Size:", vocab_size)
print(chars)

Vocabulary Size: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


Step 1.4 Character → Integer Mapping

In [ ]:
char_to_idx = {
    ch: i
    for i, ch in enumerate(chars)
}

idx_to_char = {
    i: ch
    for i, ch in enumerate(chars)
}

Step 1.5 Encode Entire Dataset

In [ ]:
encoded_text = [
    char_to_idx[ch]
    for ch in text
]

print(encoded_text[:30])

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43]


Step 1.6 Create Training Sequences

In [ ]:
sequence_length = 100

inputs = []
targets = []

for i in range(len(encoded_text) - sequence_length):

    x = encoded_text[i:i + sequence_length]

    y = encoded_text[i + 1:i + sequence_length + 1]

    inputs.append(x)
    targets.append(y)

print("Total Training Samples:", len(inputs))

Total Training Samples: 1115294


Step 1.7 Convert to PyTorch Tensors

In [ ]:
inputs = torch.tensor(inputs, dtype=torch.long)

targets = torch.tensor(targets, dtype=torch.long)

print(inputs.shape)
print(targets.shape)

torch.Size([1115294, 100])
torch.Size([1115294, 100])


**Step 2: Create Dataset and DataLoader**

Step 2.1 Create a Custom Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):

    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

Step 2.2 Create Dataset Object

In [ ]:
dataset = CharDataset(inputs, targets)

print("Dataset Size:", len(dataset))

Dataset Size: 1115294


Step 2.3 Create DataLoader

In [ ]:
batch_size = 64

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

Step 2.4 Check the DataLoader

In [ ]:
x_batch, y_batch = next(iter(train_loader))

print("Input Shape :", x_batch.shape)
print("Target Shape:", y_batch.shape)

Input Shape : torch.Size([64, 100])
Target Shape: torch.Size([64, 100])


Step 3: Build the RNN Model from Scratch


Step 3.1 Create the RNN Class

In [ ]:
class RNNFromScratch(nn.Module):

    def __init__(self, vocab_size, hidden_size):

        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.Wxh = nn.Parameter(
            torch.randn(hidden_size, vocab_size) * 0.01
        )

        self.Whh = nn.Parameter(
            torch.randn(hidden_size, hidden_size) * 0.01
        )

        self.Why = nn.Parameter(
            torch.randn(vocab_size, hidden_size) * 0.01
        )

        self.bh = nn.Parameter(
            torch.zeros(hidden_size, 1)
        )

        self.by = nn.Parameter(
            torch.zeros(vocab_size, 1)
        )

    def forward(self, x):

        batch_size = x.size(0)
        sequence_length = x.size(1)

        h = torch.zeros(
            batch_size,
            self.hidden_size,
            device=x.device
        )

        outputs = []

        for t in range(sequence_length):

            x_onehot = F.one_hot(
                x[:, t],
                num_classes=self.vocab_size
            ).float()

            h = torch.tanh(
                x_onehot @ self.Wxh.T +
                h @ self.Whh.T +
                self.bh.T
            )

            y = h @ self.Why.T + self.by.T

            outputs.append(y)

        outputs = torch.stack(outputs, dim=1)

        return outputs

Step 4.1 Create the Model

In [ ]:
hidden_size = 128

model = RNNFromScratch(
    vocab_size=vocab_size,
    hidden_size=hidden_size
).to(device)

print(model)

# Define Loss Function
criterion = nn.CrossEntropyLoss()

# Step 4.3 Define Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)




RNNFromScratch()


Step 4.2 Training Hyperparameters

In [15]:
epochs = 10
for epoch in range(epochs):

    model.train()

    total_loss = 0

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        ####################################
        # Forward
        ####################################

        outputs = model(x_batch)

        ####################################
        # Reshape for CrossEntropyLoss
        ####################################

        outputs = outputs.reshape(
            -1,
            vocab_size
        )

        targets = y_batch.reshape(-1)

        ####################################
        # Compute Loss
        ####################################

        loss = criterion(
            outputs,
            targets
        )

        ####################################
        # Backpropagation
        ####################################

        optimizer.zero_grad()

        loss.backward()

        ####################################
        # Gradient Clipping
        ####################################

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5
        )

        ####################################
        # Update Parameters
        ####################################

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss = {avg_loss:.4f}"
    )

Epoch [1/10] Loss = 1.5120


KeyboardInterrupt: 

**Step 5: Text Generation (Prediction)**

Step 5.1 Create a Generate Function

In [ ]:
def generate_text(model,
                  start_text,
                  length=300):

    model.eval()

    generated = start_text

    ##################################################
    # Initial Hidden State
    ##################################################

    h = torch.zeros(
        1,
        model.hidden_size,
        device=device
    )

    ##################################################
    # Feed Initial Text
    ##################################################

    with torch.no_grad():

        for ch in start_text:

            idx = torch.tensor(
                [[char_to_idx[ch]]],
                device=device
            )

            x = F.one_hot(
                idx,
                num_classes=vocab_size
            ).float()

            x = x.squeeze(1)

            h = torch.tanh(
                x @ model.Wxh.T +
                h @ model.Whh.T +
                model.bh.T
            )

        ##################################################
        # Start Predicting New Characters
        ##################################################

        current_char = start_text[-1]

        for _ in range(length):

            idx = torch.tensor(
                [[char_to_idx[current_char]]],
                device=device
            )

            x = F.one_hot(
                idx,
                num_classes=vocab_size
            ).float()

            x = x.squeeze(1)

            ##############################################
            # RNN Cell
            ##############################################

            h = torch.tanh(
                x @ model.Wxh.T +
                h @ model.Whh.T +
                model.bh.T
            )

            ##############################################
            # Output Layer
            ##############################################

            y = h @ model.Why.T + model.by.T

            ##############################################
            # Convert to Probabilities
            ##############################################

            probs = F.softmax(
                y,
                dim=1
            )

            ##############################################
            # Sample Next Character
            ##############################################

            next_idx = torch.multinomial(
                probs,
                1
            ).item()

            next_char = idx_to_char[next_idx]

            generated += next_char

            current_char = next_char

    return generated

Step 5.2 Generate Text

In [ ]:
print(generate_text(
    model,
    start_text="To be",
    length=500
))